# Vanilla RAG в Colab

Выберите GPU в настройках среды выполнения. Положите `articles.json`, `questions.json` и `ground_truth.json` в папку `rag-colab/data` на Google Drive. Дайте ноутбуку доступ к существующему секрету `GROQ_API_KEY`. Выполняйте ячейки сверху вниз.

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
import os
import subprocess
import sys

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/rag-colab/data")
RESULTS = Path("/content/drive/MyDrive/rag-colab/results/v1")
REPO = "https://github.com/semenovfilipp/rag.git"
BRANCH = "feature/vanilla-rag"
CHECKOUT = Path("/content/rag")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, str(CHECKOUT)], check=True)
else:
    subprocess.run(["git", "-C", str(CHECKOUT), "pull", "--ff-only", "origin", BRANCH], check=True)

def experiment(*args):
    subprocess.run([
        sys.executable, str(CHECKOUT / "scripts/experiment.py"),
        "--data", str(DATA), "--results", str(RESULTS), *args,
    ], check=True)

experiment("validate")

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "-r", str(CHECKOUT / "requirements.txt"), "vllm",
], check=True)

In [ ]:
import json
import time
from urllib.error import URLError
from urllib.request import urlopen

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LOG = Path("/content/vllm.log")

def vllm_ready():
    try:
        with urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as response:
            return any(row["id"] == MODEL for row in json.load(response)["data"])
    except (URLError, TimeoutError):
        return False

if not vllm_ready():
    if "server" not in globals() or server.poll() is not None:
        with LOG.open("w") as log:
            server = subprocess.Popen([
                "vllm", "serve", MODEL, "--host", "127.0.0.1",
                "--max-model-len", "4096", "--dtype", "half",
                "--generation-config", "vllm",
            ], stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    while not vllm_ready():
        if server.poll() is not None or time.monotonic() >= deadline:
            raise RuntimeError(LOG.read_text(errors="replace")[-6000:])
        time.sleep(3)
print("vLLM ready")

In [ ]:
# Проверяем весь путь генерации на двух вопросах. Ответы сохраняются.
for mode in ("groq", "local", "rag"):
    experiment("run", mode, "--limit", "2")

In [ ]:
for mode in ("groq", "local", "rag"):
    experiment("run", mode)

experiment("evaluate")
print(f"Результаты: {RESULTS}")